In [ ]:
import asyncio
import pandas as pd
import numpy as np
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from numba import njit
import pyotp
from NorenRestApiPy.NorenApi import NorenApi

# Assuming these functions are defined elsewhere
from .indicator_functions import (
    ema_numba, jma_numba, alma_numba, rsi_trail_numba, atr_numba,
    chandelier_exit_numba, calculate_average_height_range, calculate_current_bar_length
)

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')
        global api
        api = self

class TradingState:
    def __init__(self):
        self.ce_trading_symbol = None
        self.pe_trading_symbol = None
        self.ce_trading_token = None
        self.pe_trading_token = None

class AsyncTradingSystem:
    def __init__(self):
        self.feed_queue = asyncio.Queue()
        self.resampled_data = {}
        self.current_positions = {}
        self.state = TradingState()
        self.trading_active = True
        self.option_symbols_subscribed = False
        self.resample_frequency = '5s'
        self.range_length = 3
        self.symbol = 'CRUDEOILM'
        self.exchange = 'MCX'
        self.initial_token = '428870'
        self.subscription_string = f"{self.exchange}|{self.initial_token}"
        self.executor = ThreadPoolExecutor(max_workers=4)

    async def initialize(self):
        # Initialize API and login
        usercred = pd.read_excel("usercred.xlsx")
        user, pwd, vc, app_key, imei, qr = usercred.iloc[0]
        factor2 = pyotp.TOTP(qr).now()
        
        self.api = ShoonyaApiPy()
        ret = await asyncio.to_thread(
            self.api.login,
            userid=user, password=pwd, twoFA=factor2,
            vendor_code=vc, api_secret=app_key, imei=imei
        )
        if not ret:
            raise Exception("Login failed")

    async def start(self):
        await self.initialize()
        await asyncio.gather(
            self.connect_and_subscribe(),
            self.process_feed(),
            self.resample_ticks(),
            self.trading_logic(),
            self.update_atm_symbols()
        )

    async def connect_and_subscribe(self):
        def on_message(message):
            asyncio.run_coroutine_threadsafe(self.feed_queue.put(message), asyncio.get_event_loop())

        await asyncio.to_thread(
            self.api.start_websocket,
            order_update_callback=self.event_handler_order_update,
            subscribe_callback=on_message,
            socket_open_callback=self.on_socket_open
        )
        
        while not hasattr(self, 'feed_opened') or not self.feed_opened:
            await asyncio.sleep(1)
        
        await asyncio.to_thread(self.api.subscribe, [self.subscription_string])

    def on_socket_open(self):
        self.feed_opened = True

    def event_handler_order_update(self, update):
        print(f"Order update: {update}")

    async def process_feed(self):
        while True:
            message = await self.feed_queue.get()
            await self.process_tick_data(message)
            self.feed_queue.task_done()

    async def process_tick_data(self, tick_data):
        if 'lp' in tick_data and 'tk' in tick_data:
            timestamp = pd.Timestamp.fromtimestamp(int(tick_data['ft']))
            token = tick_data['tk']
            price = float(tick_data['lp'])

            if token not in self.resampled_data:
                self.resampled_data[token] = pd.DataFrame(columns=['open', 'high', 'low', 'close'])

            df = self.resampled_data[token]
            current_time = timestamp.floor(self.resample_frequency)

            if df.empty or df.index[-1] < current_time:
                df.loc[current_time] = [price, price, price, price]
            else:
                df.loc[current_time, 'high'] = max(df.loc[current_time, 'high'], price)
                df.loc[current_time, 'low'] = min(df.loc[current_time, 'low'], price)
                df.loc[current_time, 'close'] = price

    async def resample_ticks(self):
        while True:
            for token, df in self.resampled_data.items():
                if not df.empty:
                    df_copy = df.copy()
                    long_stop, short_stop, direction = chandelier_exit_numba(
                        df_copy['open'].values, df_copy['high'].values,
                        df_copy['low'].values, df_copy['close'].values
                    )
                    df_copy['long_stop'] = long_stop
                    df_copy['short_stop'] = short_stop
                    df_copy['ce_direction'] = direction
                    self.resampled_data[token] = df_copy.dropna(subset=['open', 'high', 'low', 'close'])
            await asyncio.sleep(1)  # Adjust as needed

    async def trading_logic(self):
        while self.trading_active:
            for token, df in self.resampled_data.items():
                if not df.empty and len(df) >= 3:
                    await self.process_token_data(token, df)
            await asyncio.sleep(1)  # Adjust as needed

    async def process_token_data(self, token, df):
        high = df['high'].values
        low = df['low'].values
        average_height, current_candle_height = calculate_average_height_range(high, low, self.range_length)
        
        current_direction = df['ce_direction'].iloc[-1]
        previous_direction = df['ce_direction'].iloc[-2]

        await self.execute_trade_logic(token, current_direction, previous_direction, df)

    async def execute_trade_logic(self, token, current_direction, previous_direction, df):
        if token in self.current_positions:
            position_data = self.current_positions[token]
            if self.should_exit_trade(position_data, current_direction, previous_direction):
                await self.exit_trade(token, position_data)
        elif self.should_enter_trade(current_direction, previous_direction):
            await self.enter_trade(token, current_direction)

    def should_exit_trade(self, position_data, current_direction, previous_direction):
        position_type = position_data["position_type"]
        return ((position_type == 'call_buy' and current_direction == -1 and previous_direction == 1) or
                (position_type == 'put_buy' and current_direction == 1 and previous_direction == -1))

    async def exit_trade(self, token, position_data):
        op_token = position_data["option_token"]
        op_symbol = position_data["symbol_name"]
        price = await self.get_latest_price_option(op_token)
        if price is not None:
            action_time = pd.Timestamp.now()
            await self.log_trade(op_symbol, "EXIT", price, action_time)
            print(f"[{action_time}] Exited {position_data['position_type']} trade for {op_symbol}")
            del self.current_positions[token]

    def should_enter_trade(self, current_direction, previous_direction):
        return ((current_direction == 1 and previous_direction == -1) or
                (current_direction == -1 and previous_direction == 1))

    async def enter_trade(self, token, current_direction):
        position_type = "call_buy" if current_direction == 1 else "put_buy"
        try:
            option_token, symbol_name = await self.get_option_details(token, position_type)
            if option_token and symbol_name:
                self.current_positions[token] = {
                    "position_type": position_type,
                    "option_token": option_token,
                    "symbol_name": symbol_name
                }
                action_time = pd.Timestamp.now()
                price = await self.get_latest_price_option(option_token)
                await self.log_trade(symbol_name, "ENTRY", price, action_time)
                print(f"[{action_time}] Entered {position_type} trade for {token}")
        except Exception as e:
            print(f"Error entering trade for {token}: {e}")

    async def get_option_details(self, token, position_type):
        # Implement logic to get option details based on the token and position type
        # This might involve querying an option chain or using predefined mappings
        # Return (option_token, symbol_name)
        pass

    async def get_latest_price_option(self, option_token):
        # Implement logic to get the latest price for the option
        # This might involve querying the API or using cached data
        pass

    async def log_trade(self, symbol, action, price, timestamp):
        # Implement trade logging logic
        pass

    async def update_atm_symbols(self):
        while True:
            try:
                result = await asyncio.to_thread(self.find_atm_strike_and_symbols)
                if result[0] is not None:
                    (self.state.ce_trading_symbol, self.state.pe_trading_symbol,
                     self.state.ce_trading_token, self.state.pe_trading_token) = result
                    if not self.option_symbols_subscribed:
                        await self.option_subscription()
                        self.option_symbols_subscribed = True
                else:
                    print("Failed to update symbols")
            except Exception as e:
                print(f"Error in update_atm_symbols: {e}")
            await asyncio.sleep(15)

    def find_atm_strike_and_symbols(self):
        # Implement the logic to find ATM strike and symbols
        # This should be a CPU-bound task suitable for running in a thread
        pass

    async def option_subscription(self):
        op_chain = await asyncio.to_thread(
            self.api.get_option_chain,
            exchange=self.exchange,
            tradingsymbol=self.state.ce_trading_symbol,
            strikeprice=self.atm_strike,
            count=2
        )
        tokens = [item['token'] for item in op_chain['values']]
        subscriptions = [f"{self.exchange}|{token}" for token in tokens]
        for sub in subscriptions:
            await asyncio.to_thread(self.api.subscribe, sub)
        print("Subscribed to:", subscriptions)

# Usage
async def main():
    trading_system = AsyncTradingSystem()
    await trading_system.start()

# if __name__ == "__main__":
#     asyncio.run(main())

loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()

# Always create a task for 'main' within the current loop
asyncio.create_task(main())

# If the loop wasn't running, start it
if not loop.is_running():
    loop.run_forever() 

In [8]:
resampled_data

{}

In [6]:
import pandas as pd
fno_scrips_mcx = pd.read_csv('//home/deep/Desktop/NEWshoonya/MCX_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
fno_scrips_mcx['StrikePrice'] = fno_scrips_mcx['StrikePrice'].astype(float)
fno_scrips_mcx.sort_values('Expiry',inplace=True)
fno_scrips_mcx.reset_index(drop=True, inplace=True)

fno_scrips = pd.read_csv('/home/deep/Desktop/NEWshoonya/NFO_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])
fno_scrips['StrikePrice'] = fno_scrips['StrikePrice'].astype(float)
fno_scrips.sort_values('Expiry',inplace=True)
fno_scrips.reset_index(drop=True, inplace=True)

/tmp/ipykernel_344/181564701.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
/tmp/ipykernel_344/181564701.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])
